# Eval-determinism gate v5 - robosuite / LIBERO

## Why v2 failed

v2 ran to completion in Colab on 2026-09-12 and **still produced no evidence**. The venv built
correctly (`CPython 3.11.16`, `numpy==1.26.4`, `mujoco 3.1.6`, `gym 0.25.2` - every pin held this
time). The probe then showed:

```
robosuite  IMPORT FAILED: ModuleNotFoundError: No module named termcolor
bddl       IMPORT FAILED: ModuleNotFoundError: No module named future
libero     IMPORT FAILED: ModuleNotFoundError: No module named libero
```

Two distinct faults:

1. **`--no-deps` left the transitive dependencies out.** robosuite needs `termcolor`, bddl needs
   `future`, and `libero.libero.benchmark` imports `torch`. I used `--no-deps` to stop the
   resolver from dragging in a conflicting numpy, and threw out every legitimate dependency
   with it.
2. **The editable install did not make `libero` importable.** `uv pip install -e` reported
   `+ libero==0.1.0 (from file:///content/LIBERO)` and the package still would not import: a
   `uv`-created venv has no setuptools path finder, so the editable hook never resolves.

**Wrong assumption, and it is the same shape as v1's:** that a package manager reporting success
means the thing is usable. v1 trusted a pin that had been silently overridden; v2 trusted an
editable install that produced nothing importable. **Both times the install step said OK and the
import step was the truth.**

## What changed in v3

- **`PYTHONPATH=/content/LIBERO` everywhere**, plus `sys.path.insert(0, "/content/LIBERO")` at the
  top of both scripts. The editable install still runs but **nothing relies on it**.
- **The transitive dependencies are installed on purpose**: `termcolor`, `future`, `hydra-core`,
  `easydict`, `h5py`, `pillow`, `matplotlib`, `cloudpickle`, `pyyaml`, `imageio`, `tqdm`, and
  **CPU-only torch** from PyTorch's CPU index.
- **An import-repair loop in the probe.** It tries `import libero.libero.envs` and
  `libero.libero.benchmark`; on `ModuleNotFoundError: No module named X` it runs
  `uv pip install --python /content/venv/bin/python X` and retries, up to 15 rounds, printing
  each round. **The gate does not run until the probe is green.**
  - It carries a **module to package alias table** (`cv2` to `opencv-python-headless`, `PIL` to
    `pillow`, `yaml` to `pyyaml`, `pkg_resources` to `setuptools`, and so on), because the failing
    module name is frequently not the PyPI name and a naive loop would spin on that.
  - It refuses to pip-install `libero` itself - that name must come from `PYTHONPATH`, and if it
    does not, the clone is the problem and the loop says so instead of masking it.
  - It bails after re-trying a package it already installed, so a genuinely broken import fails
    loudly rather than burning 15 rounds.
  - It prints what it had to install, so those can move into cell 1 next time.

Everything else is identical to v2: same T1-T5, same seeds, same K, same verdict rule.

## Pre-registered predictions - unchanged, do not edit after running

| Test | Question | **Prediction** |
|---|---|---|
| **T1** | `seed(s)` then `reset()` twice, fresh envs, identical sim state? | **PASS** |
| **T2** | reset after K steps, no reseed - does the next episode depend on K? | **PASS** |
| **T3** | same, but `seed()` called again before each reset | **PASS** |
| **T4** | 6 consecutive resets vs 5 - is the 5-run a prefix of the 6-run? | **PASS** |
| **T5** | **burn 1000 `np.random.random()` between resets** - does the next episode change? | **FAIL** |

T2 was predicted FAIL in v1 and is predicted PASS here because the gymnasium run on 2026-09-12
19:32 UTC refuted the step-count hypothesis. **T5 is the live one.**

**Verdict rule, fixed in advance:** `CONFOUND CONFIRMED` iff T1 passes and (T2 fails **or** T5
fails) - `HARNESS SOUND` iff T1, T2 and T5 all pass, **in which case P6 is dead and gets recorded
as dead** - `INCONCLUSIVE` if any control disagrees with itself - `BROKEN` if T1 fails.


## Cell 1 - uv, a Python 3.11 venv, the pinned stack, and the transitive deps

In [ ]:
import subprocess, sys, os

def sh(cmd, check=True):
    print(">>>", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print("\n".join(r.stdout.strip().splitlines()[-12:]), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError("FAILED (%d): %s" % (r.returncode, cmd))
    return r.returncode

sh("%s -m pip install -q uv" % sys.executable)
sh("uv venv /content/venv --python 3.11")

PY  = "/content/venv/bin/python"
UVP = "uv pip install --python " + PY

sh(UVP + " 'numpy==1.26.4'")
sh(UVP + " 'robosuite==1.4.0' 'bddl==1.0.1' 'gym==0.25.2' 'mujoco<3.2' "
         "easydict opencv-python-headless")

# the transitive deps v2 dropped with --no-deps
sh(UVP + " termcolor future hydra-core h5py pillow matplotlib cloudpickle "
         "pyyaml imageio tqdm")

# libero.libero.benchmark imports torch; CPU build only
sh(UVP + " torch --index-url https://download.pytorch.org/whl/cpu")

if not os.path.isdir("/content/LIBERO"):
    sh("git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO")

# performed for completeness; nothing below depends on it succeeding
sh(UVP + " -e /content/LIBERO --no-deps", check=False)

# LIBERO first-run prompt: pre-seed config so no input() is needed
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
print("\ninstall step finished - the probe decides whether it worked")

## Cell 2 - write the probe (import-repair loop + resolved versions)

In [ ]:
%%writefile /content/probe.py
"""Verify the venv can actually import LIBERO, repairing missing transitive deps as it goes."""
import sys, subprocess, importlib
sys.path.insert(0, "/content/LIBERO")

VENV = "/content/venv/bin/python"

# The failing MODULE name is often not the PyPI PACKAGE name.
ALIAS = {
    "cv2": "opencv-python-headless", "PIL": "pillow", "yaml": "pyyaml",
    "skimage": "scikit-image", "sklearn": "scikit-learn", "attr": "attrs",
    "dateutil": "python-dateutil", "google": "protobuf", "pkg_resources": "setuptools",
    "mpl_toolkits": "matplotlib", "OpenGL": "pyopengl", "egl_probe": "egl-probe",
    "Cython": "cython", "IPython": "ipython",
}
NEVER_INSTALL = {"libero"}          # on PYTHONPATH, never a PyPI package here

def uv_install(pkg):
    r = subprocess.run(["uv", "pip", "install", "--python", VENV, pkg],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    ok = r.returncode == 0
    print("      uv pip install %s -> %s" % (pkg, "ok" if ok else "FAILED"), flush=True)
    if not ok:
        for line in r.stdout.strip().splitlines()[-4:]:
            print("      " + line, flush=True)
    return ok

TARGETS = ["libero.libero.envs", "libero.libero.benchmark"]
tried, MAX = set(), 15
green = False

print("== import-repair loop ==", flush=True)
for rnd in range(1, MAX + 1):
    try:
        for t in TARGETS:
            importlib.import_module(t)
        print("  round %d: all targets import cleanly" % rnd, flush=True)
        green = True
        break
    except ModuleNotFoundError as e:
        missing = (e.name or "").split(".")[0]
        print("  round %d: missing '%s'" % (rnd, missing), flush=True)
        if not missing or missing in NEVER_INSTALL:
            print("  STOP: '%s' is not installable here (expected on "
                  "PYTHONPATH=/content/LIBERO). Check the clone." % missing, flush=True)
            sys.exit(2)
        pkg = ALIAS.get(missing, missing)
        if pkg in tried:
            print("  STOP: already installed '%s' and '%s' still will not import."
                  % (pkg, missing), flush=True)
            sys.exit(3)
        tried.add(pkg)
        uv_install(pkg)
        for mod in [m for m in sys.modules if m.startswith("libero")]:
            sys.modules.pop(mod, None)
    except Exception as e:
        print("  round %d: non-import error %s: %s" % (rnd, type(e).__name__, e), flush=True)
        raise

if not green:
    print("  STOP: still failing after %d rounds. Installed: %s" % (MAX, sorted(tried)), flush=True)
    sys.exit(4)

print("\n== resolved versions, from inside the venv ==", flush=True)
print("%-12s %s" % ("python", sys.version.split()[0]))
for m in ("numpy", "robosuite", "mujoco", "bddl", "gym", "torch", "libero"):
    try:
        mod = importlib.import_module(m)
        extra = ("  <- " + str(getattr(mod, "__file__", "?"))) if m == "libero" else ""
        print("%-12s %s%s" % (m, getattr(mod, "__version__", "installed (no __version__)"), extra))
    except Exception as e:
        print("%-12s IMPORT FAILED: %s: %s" % (m, type(e).__name__, e))

if tried:
    print("\nrepaired by installing: %s" % sorted(tried))
    print("ADD THESE TO CELL 1 so the next run does not need the loop.")
print("\nPROBE OK - safe to run the gate")


## Cell 3 - run the probe

**If this does not end in `PROBE OK`, stop.** Cells 4-5 would
produce a number about a stack that cannot import LIBERO, which is exactly how v1 and v2
each wasted a run.

In [ ]:
import subprocess, os
env = dict(os.environ, PYTHONPATH="/content/LIBERO", MPLBACKEND="Agg")
p = subprocess.run(["/content/venv/bin/python", "/content/probe.py"],
                   env=env, text=True, input="N\n", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(p.stdout)
print("probe exit code:", p.returncode)
assert p.returncode == 0, "probe failed - do not run the gate; paste the output above"

## Cell 4 - write the test script

In [ ]:
%%writefile /content/gate.py
"""T1-T5 eval-determinism gate on the real LIBERO / robosuite stack.

Runs inside the Python 3.11 venv. No policy, no checkpoint, no GPU: actions come from a
pinned RandomState and the fingerprint is LIBERO's own get_sim_state(), so any difference
in environment state is the environment.
"""
import os, sys, hashlib, json, datetime
sys.path.insert(0, "/content/LIBERO")

BACKEND = os.environ.get("MUJOCO_GL", "egl")
import numpy as np

TASK_SUITE   = "libero_object"
TASK_NAME    = "pick_up_the_alphabet_soup_and_place_it_in_the_basket"
SEEDS        = [11, 12, 13]
K_SHORT, K_LONG = 40, 80
GLOBAL_DRAWS = 1000

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

_b     = benchmark.get_benchmark_dict()[TASK_SUITE]()
_names = [_b.get_task(i).name for i in range(_b.n_tasks)]
TASK_ID = _names.index(TASK_NAME)
_task   = _b.get_task(TASK_ID)
BDDL    = os.path.join(get_libero_path("bddl_files"), _task.problem_folder, _task.bddl_file)
print("[gate] backend=%s task=%d %s" % (BACKEND, TASK_ID, _task.name), flush=True)

def mk():
    return OffScreenRenderEnv(bddl_file_name=BDDL, camera_heights=128, camera_widths=128)

def key(env):
    s = np.asarray(env.get_sim_state(), dtype=np.float64)
    return hashlib.sha256(np.ascontiguousarray(s).tobytes()).hexdigest()[:16]

def acts(k, tag, env):
    rs = np.random.RandomState(1234 if tag == "armA" else 5678)
    return [rs.uniform(-0.2, 0.2, size=env.env.action_dim) for _ in range(k)]

def seeded_once(s):
    e = mk(); e.seed(int(s)); e.reset(); k = key(e); e.close(); return k

def roll_reset(s, k, tag, reseed=None, burn=0):
    e = mk(); e.seed(int(s)); e.reset()
    for a in acts(k, tag, e):
        e.step(a)
    for _ in range(burn):
        np.random.random()                 # T5 only: advance the GLOBAL numpy RNG
    if reseed is not None:
        e.seed(int(reseed))
    e.reset()                              # unseeded unless reseed given
    kk = key(e); e.close(); return kk

def seq(s, n):
    e = mk(); e.seed(int(s)); e.reset(); out = [key(e)]
    for _ in range(n - 1):
        e.reset(); out.append(key(e))
    e.close(); return out

R = {"task": TASK_NAME, "seeds": SEEDS, "k_short": K_SHORT, "k_long": K_LONG,
     "global_draws": GLOBAL_DRAWS, "mujoco_gl": BACKEND,
     "predictions": {"T1": "PASS", "T2": "PASS", "T3": "PASS", "T4": "PASS",
                     "T5": "FAIL (pre-registered)"}}

print("\n== T1  seeded reset reproducibility ==", flush=True)
R["T1"] = {}
for s in SEEDS:
    a, b = seeded_once(s), seeded_once(s)
    R["T1"][s] = {"a": a, "b": b, "match": a == b}
    print("  seed %s: %s vs %s -> %s" % (s, a, b, "MATCH" if a == b else "DIFFER"), flush=True)
T1_PASS = all(v["match"] for v in R["T1"].values())

print("\n== T2  does the next episode depend on STEPS taken? ==", flush=True)
R["T2"] = {}
for s in SEEDS:
    sh = roll_reset(s, K_SHORT, "armA")
    lo = roll_reset(s, K_LONG,  "armB")
    ct = roll_reset(s, K_SHORT, "armA")
    R["T2"][s] = {"short": sh, "long": lo, "control": ct,
                  "arms_match": sh == lo, "control_match": sh == ct}
    print("  seed %s: K%d=%s K%d=%s arms=%s ctrl=%s"
          % (s, K_SHORT, sh, K_LONG, lo, "MATCH" if sh == lo else "DIFFER",
             "ok" if sh == ct else "UNSTABLE"), flush=True)
CONTROL_OK = all(v["control_match"] for v in R["T2"].values())
T2_PASS    = all(v["arms_match"]    for v in R["T2"].values())

print("\n== T3  does calling seed() again before each reset fix it? ==", flush=True)
R["T3"] = {}
for s in SEEDS:
    a = roll_reset(s, K_SHORT, "armA", reseed=s + 1)
    b = roll_reset(s, K_LONG,  "armB", reseed=s + 1)
    R["T3"][s] = {"a": a, "b": b, "match": a == b}
    print("  seed %s: %s vs %s -> %s" % (s, a, b, "MATCH" if a == b else "DIFFER"), flush=True)
T3_PASS = all(v["match"] for v in R["T3"].values())

print("\n== T4  6-reset vs 5-reset sequence (the mhh-gate shape) ==", flush=True)
R["T4"] = {}
for s in SEEDS:
    six, five = seq(s, 6), seq(s, 5)
    R["T4"][s] = {"six": six, "five": five, "prefix_aligned": six[:5] == five,
                  "all_distinct": len(set(six)) == len(six)}
    print("  seed %s: prefix_aligned=%s all_distinct=%s"
          % (s, six[:5] == five, len(set(six)) == len(six)), flush=True)
T4_ALIGNED = all(v["prefix_aligned"] for v in R["T4"].values())

print("\n== T5  GLOBAL numpy RNG probe -- the test this notebook exists for ==", flush=True)
R["T5"] = {}
for s in SEEDS:
    base = roll_reset(s, K_SHORT, "armA", burn=0)
    burn = roll_reset(s, K_SHORT, "armA", burn=GLOBAL_DRAWS)
    ctrl = roll_reset(s, K_SHORT, "armA", burn=0)
    R["T5"][s] = {"no_burn": base, "burned": burn, "control": ctrl,
                  "match": base == burn, "control_match": base == ctrl}
    print("  seed %s: no_burn=%s burned=%s -> %s ctrl=%s"
          % (s, base, burn,
             "MATCH (global RNG unused)" if base == burn else "DIFFER (GLOBAL RNG IS USED)",
             "ok" if base == ctrl else "UNSTABLE"), flush=True)
T5_CONTROL_OK = all(v["control_match"] for v in R["T5"].values())
T5_PASS       = all(v["match"]         for v in R["T5"].values())

import importlib
ver = {"python": sys.version.split()[0]}
for m in ("numpy", "robosuite", "mujoco", "bddl", "gym", "torch", "libero"):
    try:
        ver[m] = getattr(importlib.import_module(m), "__version__", "installed-no-__version__")
    except Exception as e:
        ver[m] = "IMPORT FAILED: %s" % type(e).__name__

if not (CONTROL_OK and T5_CONTROL_OK):
    verdict = "INCONCLUSIVE"
    reading = ("A control disagreed with itself; the harness is noisy for a reason this test "
               "does not isolate.")
elif not T1_PASS:
    verdict = "BROKEN"
    reading = "Seeded resets are not reproducible at all; suspect the install before LIBERO."
elif T2_PASS and T5_PASS:
    verdict = "HARNESS SOUND"
    reading = ("Neither step count nor global-RNG consumption changes the episode sequence. "
               "P6 is dead: record it as dead and do not revive it without a new mechanism.")
else:
    which = []
    if not T2_PASS: which.append("step count")
    if not T5_PASS: which.append("global-RNG consumption")
    verdict = "CONFOUND CONFIRMED"
    reading = ("The episode sequence depends on " + " and ".join(which) +
               ", so two arms differing in that respect are scored on different episodes. "
               "This is the paper.")

R.update({"timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
          "versions": ver, "verdict": verdict, "reading": reading,
          "T1_pass": T1_PASS, "T2_pass": T2_PASS, "T3_pass": T3_PASS,
          "T4_prefix_aligned": T4_ALIGNED, "T5_pass": T5_PASS,
          "control_ok": CONTROL_OK, "t5_control_ok": T5_CONTROL_OK})

with open("/content/det_result_robosuite.json", "w") as f:
    json.dump(R, f, indent=2, default=str)

print("\n" + "=" * 68)
print(json.dumps({k: R[k] for k in
                  ["timestamp_utc", "verdict", "reading", "versions", "T1_pass", "T2_pass",
                   "T3_pass", "T4_prefix_aligned", "T5_pass", "control_ok", "t5_control_ok",
                   "predictions"]}, indent=2))
print("=" * 68)
print("full record: /content/det_result_robosuite.json")


## Cell 5 - run the gate, EGL first then OSMesa

In [ ]:
import subprocess, os

def run_gate(backend):
    env = dict(os.environ, MUJOCO_GL=backend, PYOPENGL_PLATFORM=backend,
               PYTHONPATH="/content/LIBERO", MPLBACKEND="Agg")
    p = subprocess.run(["/content/venv/bin/python", "/content/gate.py"],
                       env=env, text=True, input="N\n", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    return p.returncode

rc = run_gate("egl")
if rc != 0:
    print("\n" + "=" * 68)
    print("EGL run failed (exit %d). Installing OSMesa and retrying once." % rc)
    print("=" * 68 + "\n")
    subprocess.run("apt-get -qq install -y libosmesa6-dev > /dev/null 2>&1", shell=True)
    rc = run_gate("osmesa")

print("\nexit code:", rc)
print("PASTE THE JSON BLOCK ABOVE BACK for Paper Choice 2026-09-12.md section 15" if rc == 0
      else "Both backends failed. Paste the traceback; this is not a result.")